# Episode 8: PPO: reinforcement learning for a GPT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep008-ppo.ipynb)

Our model learns from its own stories, scored as it writes them, with proximal policy optimisation, a critic and a penalty for drifting, and we measure it next to DPO.

[The video](https://youtu.be/8ewa70rUPME) · [The episode's page](https://mvgpt.nenuadrian.com/ep008-ppo/) · [The docs](https://nenuadrian.github.io/mvGPT/ppo/) · [The code at `ep008/end`](https://github.com/nenuadrian/mvGPT/tree/ep008/end)

This notebook runs the episode's commands on its code, as the episode left it. Models and runs from the episodes come from [the Hub](https://huggingface.co/nenuadrian/mvGPT-checkpoints), the files the videos used. What you train here comes out a little different from the video's, as training runs for a set time, on another machine. It runs on a GPU runtime, and Colab's free T4 will do.

In [ ]:
# The code as the episode left it, at its last tag, and what it needs.
import os

if not os.path.isdir("/content/mvGPT"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep008/end https://github.com/nenuadrian/mvGPT /content/mvGPT
%cd /content/mvGPT
!pip install --quiet -r requirements.txt

import torch

print(torch.cuda.get_device_name() if torch.cuda.is_available() else "No GPU: Runtime > Change runtime type > T4 GPU")

from huggingface_hub import hf_hub_download


def fetch(*names):
    # Models from the episodes, from the Hub: the files the videos used, where the scripts look for them.
    for name in names:
        print(hf_hub_download("nenuadrian/mvGPT-checkpoints", f"checkpoints/{name}/model.pt", local_dir="."))


fetch("sft", "dpo")

## The data

In [ ]:
!python data.py  # TinyStories, the 10,000-word vocabulary, and the stories as token ids

## PPO
Ten minutes with a KL weight of 0.05, then ten with 0.5.

In [ ]:
!python ppo.py 10 0.05
!python ppo.py 10 0.5
# or, to skip the twenty minutes, the video's: fetch("ppo/kl-0.05", "ppo/kl-0.5")

## Measuring them against SFT and DPO, and stories

In [ ]:
!python evaluate.py checkpoints/sft/model.pt checkpoints/dpo/model.pt checkpoints/ppo/kl-0.05/model.pt checkpoints/ppo/kl-0.5/model.pt --out results/ppo
!python sample.py checkpoints/ppo/kl-0.05/model.pt "Words: rain, cake, jump. Story:" -n 1 --seed 3
!python sample.py checkpoints/ppo/kl-0.5/model.pt "Words: rain, cake, jump. Story:" -n 1 --seed 3

Next: [episode 9, LoRA: fine-tuning a GPT with 2% of its weights](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep009-lora.ipynb)